# DGX run (2026-11-06): architecture + pipeline benchmark

This is the code that ran on the DGX A100 (`~/arch_bench/`), merged from `arch_bench.py` and `novel.py` into one readable notebook,
with the results at the end. Results files are in `../results/` (`gauss`, `blur`, `spec_eq`, `spec_c2f`).

**What was compared**
- *Architectures:* original `UNet` · path U-Nets with the old Colab bricks (`unet_v1`, `ladder_v1`) and fixed bricks (`unet_v2`, `ladder_v2`, `ladder_rev_v2`) · noise-routed U-Net (`routed_dense/fixed/learned`) · **new: SWL, the Shared-Weight Wavelet Ladder** (`swl`, `swl_wide`, `swl_p3`, `swl_long`)
- *Pipelines:* Gaussian (ε-prediction, DDIM) · blur + noise (x0-prediction, cold sampler) · **new: spectral wavelet schedules** `spec_eq`, `spec_c2f`
- *Protocol:* every model sees the identical training stream; 4,000 steps; FID/IS on 5,000 samples with the same seed

Most of the code is shared: the 7 path U-Nets are one class (`PathUNet`) with different routes, the 3 routed models one class, the 4 SWL variants one class, and training/sampling/FID are written once.

## 1. Setup and configuration

In [ ]:
import argparse, gc, glob, json, os, sys, time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

ROOT = os.environ.get("BENCH_ROOT", os.path.expanduser("~/arch_bench"))
DATA = os.environ.get("BENCH_DATA", os.path.expanduser("~/diffusion-3/cifar10_airplanes.npz"))
timesteps = 1000
embedding_dim = 128
BATCH = 128
ROUTE_COUNTS = [(0, 100, 13), (100, 250, 19), (250, 400, 19), (400, 1000, 77)]   # sums to 128
assert sum(c for *_, c in ROUTE_COUNTS) == BATCH

## 2. Original notebook code (verbatim): cosine schedule, sinusoidal encoding, the original `UNet`

In [ ]:
# ---- original notebook code (verbatim)
def cosine_schedule(timesteps, s=0.008):
    """Nichol & Dhariwal cosine schedule.

    The linear schedule destroys most of the signal in the first ~200 steps
    at 32x32, so a lot of the model's capacity is spent on timesteps that
    carry no information. The cosine schedule spreads the damage evenly.
    """
    steps = np.arange(timesteps + 1)
    f = np.cos(((steps / timesteps) + s) / (1 + s) * (np.pi / 2)) ** 2
    alpha_bar = f / f[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return np.clip(betas, 0.0001, 0.999)

def timestep_positional_encoding(timesteps, embedding_dim):
    pe = np.zeros((timesteps, embedding_dim))
    position = np.arange(0, timesteps, 1)
    power = np.arange(0, embedding_dim, 2) / embedding_dim
    pe[:, 0::2] = np.sin(position[:, np.newaxis] / (10000 ** power))
    pe[:, 1::2] = np.cos(position[:, np.newaxis] / (10000 ** power))
    return pe

class ResBlock(layers.Layer):

    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = layers.Conv2D(
            out_channels, 3, padding="same"
        )
        # GroupNorm, not BatchNorm: every image in the batch carries a
        # different noise level, so batch statistics are meaningless here.
        self.norm1 = layers.GroupNormalization(groups=8)

        self.conv2 = layers.Conv2D(
            out_channels, 3, padding="same"
        )
        self.norm2 = layers.GroupNormalization(groups=8)

        self.time_proj = layers.Dense(out_channels)

        self.activation = layers.Activation("swish")

        if in_channels != out_channels:
            self.skip = layers.Conv2D(
                out_channels, 1, padding="same"
            )
        else:
            self.skip = None

    def call(self, x, t_emb):

        if self.skip is not None:
            residual = self.skip(x)
        else:
            residual = x

        h = self.conv1(x)
        h = self.norm1(h)

        time = self.time_proj(t_emb)
        time = time[:, tf.newaxis, tf.newaxis, :]

        h = h + time
        h = self.activation(h)

        h = self.conv2(h)
        h = self.norm2(h)

        h = h + residual
        h = self.activation(h)

        return h

class AttentionBlock(layers.Layer):
    """Self-attention over the 8x8 feature map.

    Convolutions only see a local window; one attention layer at the
    bottleneck lets every pixel look at every other pixel, which is what
    keeps a generated object globally coherent instead of locally plausible.
    """

    def __init__(self, channels, num_heads=4):
        super().__init__()

        self.norm = layers.GroupNormalization(groups=8)

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=channels // num_heads
        )

    def call(self, x):

        b = tf.shape(x)[0]
        h, w, c = x.shape[1], x.shape[2], x.shape[3]

        y = self.norm(x)
        y = tf.reshape(y, (b, h * w, c))
        y = self.attention(y, y)
        y = tf.reshape(y, (b, h, w, c))

        return x + y

class DownBlock(layers.Layer):

    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = layers.Conv2D(
            out_channels,
            3,
            strides=2,
            padding="same"
        )

    def call(self, x, t_emb):

        x = self.resblock(x, t_emb)

        skip = x

        x = self.downsample(x)

        return x, skip

class UpBlock(layers.Layer):

    def __init__(self, in_channels, skip_channels, out_channels, time_dim):
        super().__init__()

        self.upsample = layers.Conv2DTranspose(
            out_channels,
            3,
            strides=2,
            padding="same"
        )

        self.resblock = ResBlock(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def call(self, x, skip, t_emb):

        x = self.upsample(x)

        x = tf.concat([x, skip], axis=-1)

        x = self.resblock(x, t_emb)

        return x

class UNet(tf.keras.Model):

    def __init__(self, input_shape, time_dim=128, base=32):
        super().__init__()

        self.time_dim = time_dim

        c1, c2, c3 = base, base * 2, base * 4

        # Let the network reshape the sinusoidal encoding itself
        self.time_mlp = tf.keras.Sequential([
            layers.Dense(time_dim * 2, activation="swish"),
            layers.Dense(time_dim * 2),
        ])

        # Input
        self.input_conv = layers.Conv2D(
            c1,
            3,
            padding="same"
        )

        # Downsampling
        self.down1 = DownBlock(
            c1, c2, time_dim
        )

        self.down2 = DownBlock(
            c2, c3, time_dim
        )

        # Bottleneck
        self.bottleneck1 = ResBlock(
            c3, c3, time_dim
        )

        self.attention = AttentionBlock(c3)

        self.bottleneck2 = ResBlock(
            c3, c3, time_dim
        )

        # Upsampling
        self.up2 = UpBlock(
            c3, c3, c2, time_dim
        )

        self.up1 = UpBlock(
            c2, c2, c1, time_dim
        )

        # Output
        self.output_conv = layers.Conv2D(
            3,
            3,
            padding="same"
        )

    def call(self, inputs):

        x, t_emb = inputs

        t_emb = self.time_mlp(t_emb)

        # Input
        x = self.input_conv(x)

        # Down
        x, skip1 = self.down1(x, t_emb)
        x, skip2 = self.down2(x, t_emb)

        # Bottleneck
        x = self.bottleneck1(x, t_emb)
        x = self.attention(x)
        x = self.bottleneck2(x, t_emb)

        # Up
        x = self.up2(x, skip2, t_emb)
        x = self.up1(x, skip1, t_emb)

        # Output
        x = self.output_conv(x)

        return x

## 3. Noise schedule and the identical training stream
`stream_batch(x, step)` depends only on the step number, so every model sees the same images, flips, timesteps and noise. Timesteps are stratified (13/19/19/77 per noise range) so the routed U-Net can split each batch with static shapes.

In [ ]:
# ---------------------------------------------------------------- schedule / data
beta_schedule = cosine_schedule(timesteps)
ALPHA_BAR_NP = np.cumprod(1.0 - beta_schedule).astype(np.float32)
ALPHA_BAR = tf.constant(ALPHA_BAR_NP)
PE = tf.constant(timestep_positional_encoding(timesteps, embedding_dim), dtype=tf.float32)


def load_data():
    d = np.load(DATA)
    x = d["x_train"].astype(np.float32) / 127.5 - 1.0
    return x


def stream_batch(x_all, step, seed=0):
    spe = len(x_all) // BATCH
    epoch, i = divmod(step, spe)
    perm = np.random.default_rng([seed, 0, epoch]).permutation(len(x_all))
    x = x_all[perm[i * BATCH:(i + 1) * BATCH]].copy()
    rng = np.random.default_rng([seed, 1, step])
    flip = rng.random(BATCH) < 0.5
    x[flip] = x[flip][:, :, ::-1, :]
    t = np.concatenate([rng.integers(lo, hi, c) for lo, hi, c in ROUTE_COUNTS]).astype("int32")
    noise = rng.standard_normal(x.shape, dtype=np.float32)
    return x, t, noise

## 4. Blur + noise degradation (verbatim from the notebook)

In [ ]:
# ---- blur + noise degradation (verbatim)
SIGMA_MAX = 2.5
KERNEL_SIZE = 17


def gaussian_kernel_2d(sigma, size):
    ax = np.arange(-(size // 2), size // 2 + 1, dtype=np.float32)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2.0 * max(sigma, 1e-3) ** 2))
    kernel /= kernel.sum()
    return kernel


# Blur sigma ramps up on the same schedule as the noise (alpha_bar), so both
# forms of corruption reach full strength together at the last timestep.
blur_sigma = SIGMA_MAX * (1.0 - np.sqrt(ALPHA_BAR_NP))

BLUR_KERNELS = tf.constant(
    np.stack([gaussian_kernel_2d(s, KERNEL_SIZE) for s in blur_sigma]),
    dtype=tf.float32
)  # shape: (timesteps, KERNEL_SIZE, KERNEL_SIZE)



def apply_blur(x, t_indices):
    """Blur every image in the batch by its own per-example kernel (indexed
    by t), in a single depthwise convolution.

    The trick: fold the batch dimension into the channel dimension so each
    example gets its own filter, since tf.nn.depthwise_conv2d applies one
    filter per input channel.
    """
    b = tf.shape(x)[0]
    h, w, c = x.shape[1], x.shape[2], x.shape[3]

    kernels = tf.gather(BLUR_KERNELS, t_indices)                    # [B, K, K]
    kernels = tf.repeat(kernels[:, :, :, tf.newaxis], c, axis=-1)   # [B, K, K, C]
    kernels = tf.transpose(kernels, [1, 2, 0, 3])                   # [K, K, B, C]
    kernels = tf.reshape(kernels, [KERNEL_SIZE, KERNEL_SIZE, b * c, 1])

    x_r = tf.transpose(x, [1, 2, 0, 3])                             # [H, W, B, C]
    x_r = tf.reshape(x_r, [1, h, w, b * c])

    blurred = tf.nn.depthwise_conv2d(
        x_r, kernels, strides=[1, 1, 1, 1], padding="SAME"
    )                                                                # [1, H, W, B*C]

    blurred = tf.reshape(blurred, [h, w, b, c])
    blurred = tf.transpose(blurred, [2, 0, 1, 3])                   # [B, H, W, C]
    return blurred


def degrade_blur_noise(x0, t_indices, noise):
    """Combined corruption: blur x0 by the timestep's sigma, then blend with
    noise using the same alpha_bar weighting as the Gaussian pipeline."""
    alpha_bar_t = tf.gather(ALPHA_BAR, t_indices)[:, tf.newaxis, tf.newaxis, tf.newaxis]
    blurred = apply_blur(x0, t_indices)
    return tf.sqrt(alpha_bar_t) * blurred + tf.sqrt(1.0 - alpha_bar_t) * noise

## 5. Path U-Net: one class, any route
`v2=False` = the Colab bricks (2×2 non-overlapping downsampling); `v2=True` = fixed bricks (3×3 stride-2 conv, avg-pool for jumps).

In [ ]:
# ---------------------------------------------------------------- path U-Net (v1 = Colab bricks, v2 = fixed bricks)
def timestep_embedding(t, dim):
    t = tf.cast(tf.reshape(t, (-1, 1)), tf.float32)
    half = dim // 2
    freqs = tf.exp(-np.log(10000.0) * tf.range(half, dtype=tf.float32) / half)
    args = t * freqs[tf.newaxis, :]
    return tf.concat([tf.sin(args), tf.cos(args)], axis=-1)


class PResBlock(layers.Layer):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.norm1 = layers.GroupNormalization(groups=8)
        self.conv1 = layers.Conv2D(out_ch, 3, padding="same")
        self.time_proj = layers.Dense(out_ch)
        self.norm2 = layers.GroupNormalization(groups=8)
        self.conv2 = layers.Conv2D(out_ch, 3, padding="same")
        self.skip = layers.Conv2D(out_ch, 1) if in_ch != out_ch else None

    def call(self, x, t_emb):
        h = self.conv1(tf.nn.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, tf.newaxis, tf.newaxis, :]
        h = self.conv2(tf.nn.silu(self.norm2(h)))
        return h + (self.skip(x) if self.skip is not None else x)


class PAttentionBlock(layers.Layer):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = layers.GroupNormalization(groups=8)
        self.attention = layers.MultiHeadAttention(num_heads, channels // num_heads)

    def call(self, x):
        b = tf.shape(x)[0]
        h, w, c = x.shape[1], x.shape[2], x.shape[3]
        y = tf.reshape(self.norm(x), (b, h * w, c))
        y = self.attention(y, y)
        return x + tf.reshape(y, (b, h, w, c))


class Resample(layers.Layer):
    """v1: down k levels = conv kernel=stride=2**k (non-overlapping).
       v2: down = 3x3 stride-2 conv for one level, avg-pool + 3x3 conv for bigger jumps."""
    def __init__(self, from_level, to_level, out_ch, v2):
        super().__init__()
        k = to_level - from_level
        if k > 0 and not v2:
            self.op = layers.Conv2D(out_ch, 2 ** k, strides=2 ** k)
        elif k == 1:
            self.op = layers.Conv2D(out_ch, 3, strides=2, padding="same")
        elif k > 1:
            self.op = tf.keras.Sequential([layers.AveragePooling2D(2 ** k), layers.Conv2D(out_ch, 3, padding="same")])
        else:
            self.op = tf.keras.Sequential([layers.UpSampling2D(2 ** (-k), interpolation="nearest"),
                                           layers.Conv2D(out_ch, 3, padding="same")])

    def call(self, x):
        return self.op(x)


class PathUNet(tf.keras.Model):
    def __init__(self, path, base=32, mults=(1, 2, 2, 4), time_dim=128, attn_levels=(3,), v2=False):
        super().__init__()
        self.levels = list(path)
        self.time_dim = time_dim
        ch = [base * m for m in mults]
        self.time_mlp = tf.keras.Sequential([layers.Dense(time_dim * 4, activation="swish"), layers.Dense(time_dim * 4)])
        self.input_conv = layers.Conv2D(ch[0], 3, padding="same")
        self.moves, self.blocks, self.attns = [], [], []
        seen = {0: ch[0]}
        cur = 0
        self.blocks.append(PResBlock(ch[0], ch[0]))
        for nxt in self.levels[1:]:
            self.moves.append(Resample(cur, nxt, ch[nxt], v2))
            self.blocks.append(PResBlock(ch[nxt] + seen.get(nxt, 0), ch[nxt]))
            self.attns.append(PAttentionBlock(ch[nxt]) if nxt in attn_levels else None)
            seen[nxt] = ch[nxt]
            cur = nxt
        self.out_norm = layers.GroupNormalization(groups=8)
        self.output_conv = layers.Conv2D(3, 3, padding="same")

    def call(self, inputs, training=False):
        x, t = inputs
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim))
        x = self.blocks[0](self.input_conv(x), t_emb)
        memory = {0: x}
        for nxt, move, block, attn in zip(self.levels[1:], self.moves, self.blocks[1:], self.attns):
            x = move(x)
            if nxt in memory:
                x = tf.concat([x, memory[nxt]], axis=-1)
            x = block(x, t_emb)
            if attn is not None:
                x = attn(x)
            memory[nxt] = x
        return self.output_conv(tf.nn.silu(self.out_norm(x)))

## 6. Noise-routed U-Net
Two blocks (a, b) per level going down and up; which blocks run depends on t (`ROUTES`). `dense` = all blocks, `fixed` = the routing table, `learned` = a gate per block.

In [ ]:
# ---------------------------------------------------------------- noise-routed U-Net
ROUTES = [(0, 100, {0: "ab", 1: "ab", 2: "", 3: ""}),
          (100, 250, {0: "", 1: "a", 2: "a", 3: "a"}),
          (250, 400, {0: "", 1: "", 2: "ab", 3: "a"}),
          (400, 1000, {0: "", 1: "", 2: "a", 3: "ab"})]
DENSE_ROUTE = {0: "ab", 1: "ab", 2: "ab", 3: "ab"}
ROUTED_BLOCKS = [f"{s}{l}{k}" for s in "ed" for l in range(4) for k in "ab"]
LEVEL_COST = {0: 1.0, 1: 1.0, 2: 0.33, 3: 0.25}
BLOCK_COST = np.array([2 * LEVEL_COST[int(b[1])] for b in ROUTED_BLOCKS], np.float32)
GATE_PENALTY = 0.01


def route_cost(route):
    return sum(2 * LEVEL_COST[l] * len(route[l]) for l in range(4)) / BLOCK_COST.sum()


class RoutedUNet(tf.keras.Model):
    def __init__(self, mode, base=32, mults=(1, 2, 2, 4), time_dim=128):
        super().__init__()
        self.mode = mode
        self.time_dim = time_dim
        ch = [base * m for m in mults]
        self.time_mlp = tf.keras.Sequential([layers.Dense(time_dim * 4, activation="swish"), layers.Dense(time_dim * 4)])
        self.stem = layers.Conv2D(ch[0], 3, padding="same")
        self.downs = [layers.Conv2D(ch[l], 3, strides=2, padding="same") for l in (1, 2, 3)]
        self.ups = [tf.keras.Sequential([layers.UpSampling2D(2, interpolation="nearest"),
                                         layers.Conv2D(ch[l - 1], 3, padding="same")]) for l in (1, 2, 3)]
        self.fuse = [layers.Conv2D(ch[l], 1) for l in range(4)]
        self.blocks = {n: PResBlock(ch[int(n[1])], ch[int(n[1])]) for n in ROUTED_BLOCKS}
        self.attns = {n: PAttentionBlock(ch[2]) for n in ROUTED_BLOCKS if n[1] == "2"}
        self.mid1 = PResBlock(ch[3], ch[3])
        self.mid_attn = PAttentionBlock(ch[3])
        self.mid2 = PResBlock(ch[3], ch[3])
        if mode == "learned":
            self.gate_head = layers.Dense(len(ROUTED_BLOCKS), kernel_initializer="zeros",
                                          bias_initializer=tf.keras.initializers.Constant(2.0))
        self.out_norm = layers.GroupNormalization(groups=8)
        self.out_conv = layers.Conv2D(3, 3, padding="same")

    def gate_values(self, t):
        return tf.sigmoid(self.gate_head(self.time_mlp(timestep_embedding(t, self.time_dim))))

    def forward(self, x, t, route=None, active=None, return_gates=False):
        route = route or DENSE_ROUTE
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim))
        gates = tf.sigmoid(self.gate_head(t_emb)) if self.mode == "learned" else None

        def run(name, h):
            if name[2] not in route[int(name[1])] or (active is not None and name not in active):
                return h
            y = self.blocks[name](h, t_emb)
            if name in self.attns:
                y = self.attns[name](y)
            if gates is not None:
                g = gates[:, ROUTED_BLOCKS.index(name)][:, None, None, None]
                y = h + g * (y - h)
            return y

        h = self.stem(x)
        skips = []
        for level in range(4):
            if level > 0:
                h = self.downs[level - 1](h)
            for k in "ab":
                h = run(f"e{level}{k}", h)
            skips.append(h)
        h = self.mid2(self.mid_attn(self.mid1(h, t_emb)), t_emb)
        for level in (3, 2, 1, 0):
            h = self.fuse[level](tf.concat([h, skips[level]], axis=-1))
            for k in "ab":
                h = run(f"d{level}{k}", h)
            if level > 0:
                h = self.ups[level - 1](h)
        out = self.out_conv(tf.nn.silu(self.out_norm(h)))
        return (out, gates) if return_gates else out

    def call(self, inputs, training=False):
        x, t = inputs
        return self.forward(x, t)

## 7. Model registry and loss

In [ ]:
# ---------------------------------------------------------------- model registry
V2 = dict(base=32, mults=(2, 4, 4), attn_levels=(2,), v2=True)      # 64/128/128 ch, stop at 8x8, attention at 8x8
MODELS = {
    "original_unet":  ("orig", None),
    "unet_v1":        ("path", dict(path=[0, 1, 2, 3, 2, 1, 0], base=32)),
    "ladder_v1":      ("path", dict(path=[0, 1, 0, 1, 2, 1, 0, 1, 2, 3, 2, 1, 0], base=32)),
    "unet_v2":        ("path", dict(path=[0, 1, 2, 1, 0], **V2)),
    "ladder_v2":      ("path", dict(path=[0, 1, 0, 1, 2, 1, 0], **V2)),
    "ladder_rev_v2":  ("path", dict(path=[0, 1, 2, 1, 0, 1, 0], **V2)),
    "wnet_v2":        ("path", dict(path=[0, 1, 2, 1, 0, 1, 2, 1, 0], **V2)),
    "routed_dense":   ("routed", "dense"),
    "routed_fixed":   ("routed", "fixed"),
    "routed_learned": ("routed", "learned"),
}


def build(name):
    tf.keras.utils.set_random_seed(0)
    kind, cfg = MODELS[name]
    if kind == "orig":
        net = UNet(input_shape=(32, 32, 3), time_dim=embedding_dim, base=32)
        fwd = lambda x, t: net([x, tf.gather(PE, t)])
    elif kind == "path":
        net = PathUNet(**cfg)
        fwd = lambda x, t: net([x, t])
    else:
        net = RoutedUNet(cfg)
        fwd = lambda x, t: net.forward(x, t)
        net([tf.zeros((2, 32, 32, 3)), tf.constant([0, 999], tf.int32)])
    fwd(tf.zeros((2, 32, 32, 3)), tf.constant([0, 999], tf.int32))
    return net, fwd


def model_loss(name, net, fwd, x_in, t, target):
    """Returns (total, mse). routed_fixed sends each fixed-size route slice through its route."""
    if name == "routed_fixed":
        sq, i = 0.0, 0
        for (lo, hi, c), (_, _, route) in zip(ROUTE_COUNTS, ROUTES):
            pred = net.forward(x_in[i:i + c], t[i:i + c], route=route)
            sq += tf.reduce_sum(tf.square(pred - target[i:i + c]))
            i += c
        mse = sq / tf.cast(tf.size(target), tf.float32)
        return mse, mse
    if name == "routed_learned":
        pred, gates = net.forward(x_in, t, return_gates=True)
        mse = tf.reduce_mean(tf.square(pred - target))
        used = tf.reduce_mean(tf.reduce_sum(gates * BLOCK_COST, axis=1)) / BLOCK_COST.sum()
        return mse + GATE_PENALTY * used, mse
    mse = tf.reduce_mean(tf.square(fwd(x_in, t) - target))
    return mse, mse


def run_dir(kind, name):
    return os.path.join(ROOT, kind, name)


def make_state(name):
    net, fwd = build(name)
    opt = tf.keras.optimizers.Adam(learning_rate=2e-4)
    opt.build(net.trainable_variables)
    ema = [tf.Variable(v, trainable=False) for v in net.trainable_variables]
    ema_n = tf.Variable(0.0, trainable=False)
    step = tf.Variable(0, dtype=tf.int64, trainable=False)
    ckpt = tf.train.Checkpoint(model=net, optimizer=opt, ema_weights=ema, ema_step=ema_n, step=step)
    return net, fwd, opt, ema, ema_n, step, ckpt

## 8. Training
`train` = one model per process; `train_multi` = all models in one process, one step each in turn (identical step counts; used on the shared GPU).

In [ ]:
# ---------------------------------------------------------------- train
def train(name, kind, steps, ckpt_every=500):
    d = run_dir(kind, name)
    os.makedirs(d, exist_ok=True)
    net, fwd, opt, ema, ema_n, step, ckpt = make_state(name)
    manager = tf.train.CheckpointManager(ckpt, os.path.join(d, "ckpt"), max_to_keep=1)
    if manager.latest_checkpoint:
        ckpt.restore(manager.latest_checkpoint).expect_partial()
    start = int(step.numpy())
    x_all = load_data()
    print(f"[{kind}/{name}] params {net.count_params():,}  steps {start} -> {steps}", flush=True)

    @tf.function(jit_compile=(name != "routed_fixed"), reduce_retracing=True)
    def step_fn(x_0, t, noise):
        a = tf.gather(ALPHA_BAR, t)[:, None, None, None]
        if kind == "gauss":
            x_t, target = tf.sqrt(a) * x_0 + tf.sqrt(1.0 - a) * noise, noise
        else:
            x_t, target = degrade_blur_noise(x_0, t, noise), x_0
        with tf.GradientTape() as tape:
            total, mse = model_loss(name, net, fwd, x_t, t, target)
        grads = tape.gradient(total, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        ema_n.assign_add(1.0)
        dd = tf.minimum(0.999, (1.0 + ema_n) / (10.0 + ema_n))
        for e, v in zip(ema, net.trainable_variables):
            e.assign(dd * e + (1.0 - dd) * v)
        return mse

    log = open(os.path.join(d, "loss.csv"), "a")
    losses, t0 = [], time.time()
    target_file = os.path.join(ROOT, "target_steps")
    s = start
    while True:
        if s % 50 == 0 and os.path.exists(target_file):
            try:
                steps = int(open(target_file).read().strip())
            except ValueError:
                pass
        if s >= steps:
            break
        x, t, n = stream_batch(x_all, s)
        losses.append(float(step_fn(x, t, n)))
        step.assign(s + 1)
        if (s + 1) % ckpt_every == 0:
            manager.save()
            m = float(np.mean(losses[-ckpt_every:]))
            log.write(f"{s + 1},{m:.6f}\n"); log.flush()
            print(f"[{kind}/{name}] step {s + 1} loss {m:.4f}  {(time.time() - t0) / len(losses) * 1000:.0f} ms/step", flush=True)
        s += 1
    manager.save()
    json.dump(dict(params=int(net.count_params()), steps=int(step.numpy()),
                   ms_per_step=(time.time() - t0) / max(1, len(losses)) * 1000), open(os.path.join(d, "train_done.json"), "w"))


def train_multi(names, kind, steps, ckpt_every=500):
    """Train several models in ONE process, one step each in turn: identical step counts by construction."""
    x_all = load_data()
    runs = []
    for name in names:
        d = run_dir(kind, name)
        os.makedirs(d, exist_ok=True)
        net, fwd, opt, ema, ema_n, step, ckpt = make_state(name)
        manager = tf.train.CheckpointManager(ckpt, os.path.join(d, "ckpt"), max_to_keep=1)
        if manager.latest_checkpoint:
            ckpt.restore(manager.latest_checkpoint).expect_partial()

        def make(name=name, net=net, fwd=fwd, opt=opt, ema=ema, ema_n=ema_n):
            @tf.function(jit_compile=(name != "routed_fixed"), reduce_retracing=True)
            def step_fn(x_0, t, noise):
                a = tf.gather(ALPHA_BAR, t)[:, None, None, None]
                if kind == "gauss":
                    x_t, target = tf.sqrt(a) * x_0 + tf.sqrt(1.0 - a) * noise, noise
                else:
                    x_t, target = degrade_blur_noise(x_0, t, noise), x_0
                with tf.GradientTape() as tape:
                    total, mse = model_loss(name, net, fwd, x_t, t, target)
                grads = tape.gradient(total, net.trainable_variables)
                opt.apply_gradients(zip(grads, net.trainable_variables))
                ema_n.assign_add(1.0)
                dd = tf.minimum(0.999, (1.0 + ema_n) / (10.0 + ema_n))
                for e, v in zip(ema, net.trainable_variables):
                    e.assign(dd * e + (1.0 - dd) * v)
                return mse
            return step_fn
        runs.append(dict(name=name, d=d, net=net, step=step, manager=manager, fn=make(), losses=[], secs=0.0,
                         log=open(os.path.join(d, "loss.csv"), "a")))
        print(f"[{kind}/{name}] params {net.count_params():,}  at step {int(step.numpy())}", flush=True)
    s = min(int(r["step"].numpy()) for r in runs)
    target_file = os.path.join(ROOT, "target_steps")
    while True:
        if s % 50 == 0 and os.path.exists(target_file):
            try:
                steps = int(open(target_file).read().strip())
            except ValueError:
                pass
        if s >= steps:
            break
        x, t, n = stream_batch(x_all, s)
        xt, tt, nt = tf.constant(x), tf.constant(t), tf.constant(n)
        for r in runs:
            if int(r["step"].numpy()) > s:
                continue
            t0 = time.time()
            r["losses"].append(float(r["fn"](xt, tt, nt)))
            r["secs"] += time.time() - t0
            r["step"].assign(s + 1)
        s += 1
        if s % ckpt_every == 0:
            for r in runs:
                r["manager"].save()
                m = float(np.mean(r["losses"][-ckpt_every:]))
                r["log"].write(f"{s},{m:.6f}\n"); r["log"].flush()
                print(f"[{kind}/{r['name']}] step {s} loss {m:.4f}  {r['secs'] / max(1, len(r['losses'])) * 1000:.0f} ms/step", flush=True)
    for r in runs:
        r["manager"].save()
        json.dump(dict(params=int(r["net"].count_params()), steps=int(r["step"].numpy()),
                       ms_per_step=r["secs"] / max(1, len(r["losses"])) * 1000), open(os.path.join(r["d"], "train_done.json"), "w"))

## 9. Evaluation: DDIM / cold sampler, Inception FID and IS, report

In [ ]:
# ---------------------------------------------------------------- eval


N_EVAL = int(os.environ.get("BENCH_N_EVAL", 5000)); EVAL_STEPS = int(os.environ.get("BENCH_EVAL_STEPS", 100))
EVAL_BATCH, EVAL_SEED = min(500, N_EVAL), 1234
SAMPLE_TS = np.linspace(0, timesteps - 1, EVAL_STEPS).round().astype(int)


class InceptionScorer:
    def __init__(self):
        base = tf.keras.applications.InceptionV3(include_top=True, weights="imagenet")
        self.model = tf.keras.Model(base.input, [base.get_layer("avg_pool").output, base.output])
        self.f = tf.function(lambda x: self.model(tf.image.resize(x, (299, 299)) * 2.0 - 1.0, training=False))

    def __call__(self, images_01, batch=100):
        feats, probs = [], []
        for i in range(0, len(images_01), batch):
            f, p = self.f(tf.convert_to_tensor(images_01[i:i + batch], tf.float32))
            feats.append(f.numpy()); probs.append(p.numpy())
        return np.concatenate(feats).astype(np.float64), np.concatenate(probs).astype(np.float64)


def fid_from_features(feats, mu_ref, sigma_ref):
    import scipy.linalg
    mu, sigma = feats.mean(axis=0), np.cov(feats, rowvar=False)
    diff = mu - mu_ref
    covmean = scipy.linalg.sqrtm(sigma.dot(sigma_ref))
    if not np.isfinite(covmean).all():
        off = np.eye(sigma.shape[0]) * 1e-6
        covmean = scipy.linalg.sqrtm((sigma + off).dot(sigma_ref + off))
    return float(diff.dot(diff) + np.trace(sigma) + np.trace(sigma_ref) - 2.0 * np.trace(np.real(covmean)))

In [ ]:
def inception_score(probs, splits=10):
    s = []
    for part in np.array_split(probs, splits):
        py = part.mean(axis=0, keepdims=True)
        s.append(np.exp((part * (np.log(part + 1e-12) - np.log(py + 1e-12))).sum(axis=1).mean()))
    return float(np.mean(s)), float(np.std(s))


def reference(scorer):
    p = os.path.join(ROOT, "fid_reference_airplanes.npz")
    if not os.path.exists(p):
        f, pr = scorer(np.clip((load_data() + 1) / 2, 0, 1))
        m, is_ = inception_score(pr)
        np.savez(p, mu=f.mean(0), sigma=np.cov(f, rowvar=False), is_mean=m, is_std=is_)
    r = np.load(p)
    return r["mu"], r["sigma"], (float(r["is_mean"]), float(r["is_std"]))


def predictors(name, net, fwd):
    """Yields (label, predict_fn(x, t_int_scalar) , block-compute fraction or None)."""
    if name == "routed_fixed":
        per = [tf.function(lambda x, t, r=r: net.forward(x, t, route=r)) for *_, r in ROUTES]
        def p(x, tv):
            i = next(i for i, (lo, hi, _) in enumerate(ROUTES) if lo <= tv < hi or i == len(ROUTES) - 1)
            return per[i](x, tf.fill([tf.shape(x)[0]], tv))
        bc = float(np.mean([route_cost(next(r for lo, hi, r in ROUTES if lo <= tv < hi or hi == 1000)) for tv in SAMPLE_TS]))
        yield name, p, bc
    elif name == "routed_learned":
        f_all = tf.function(lambda x, t: net.forward(x, t))
        yield name + "_all", (lambda x, tv: f_all(x, tf.fill([tf.shape(x)[0]], tv))), 1.0
        gates = net.gate_values(tf.constant(SAMPLE_TS, tf.int32)).numpy()
        act = {int(tv): frozenset(b for b, g in zip(ROUTED_BLOCKS, row) if g >= 0.1) for tv, row in zip(SAMPLE_TS, gates)}
        cache = {}
        def p(x, tv):
            a = act[tv]
            if a not in cache:
                cache[a] = tf.function(lambda x, t, a=a: net.forward(x, t, active=a))
            return cache[a](x, tf.fill([tf.shape(x)[0]], tv))
        bc = float(np.mean([BLOCK_COST[[b in act[int(tv)] for b in ROUTED_BLOCKS]].sum() / BLOCK_COST.sum() for tv in SAMPLE_TS]))
        np.save(os.path.join(ROOT, "learned_gates.npy"), net.gate_values(tf.constant(np.arange(0, 1000, 10), tf.int32)).numpy())
        yield name + "_pruned", p, bc
    else:
        f = tf.function(lambda x, t: fwd(x, t))
        yield name, (lambda x, tv: f(x, tf.fill([tf.shape(x)[0]], tv))), (1.0 if name == "routed_dense" else None)

In [ ]:
def generate(kind, predict):
    rng = np.random.default_rng(EVAL_SEED)
    idx = SAMPLE_TS[::-1]
    out = []
    for start in range(0, N_EVAL, EVAL_BATCH):
        n = min(EVAL_BATCH, N_EVAL - start)
        z = tf.constant(rng.standard_normal((n, 32, 32, 3)), tf.float32)
        x_t = z
        for i, t in enumerate(idx):
            tp = idx[i + 1] if i + 1 < len(idx) else -1
            if kind == "gauss":
                a_t = ALPHA_BAR[t]; a_p = ALPHA_BAR[tp] if tp >= 0 else tf.constant(1.0)
                eps = predict(x_t, int(t))
                x0 = tf.clip_by_value((x_t - tf.sqrt(1.0 - a_t) * eps) / tf.sqrt(a_t), -1.0, 1.0)
                x_t = tf.sqrt(a_p) * x0 + tf.sqrt(1.0 - a_p) * eps
            else:
                x0 = tf.clip_by_value(predict(x_t, int(t)), -1.0, 1.0)
                if tp >= 0:
                    x_t = x_t - degrade_blur_noise(x0, tf.fill([n], int(t)), z) + degrade_blur_noise(x0, tf.fill([n], int(tp)), z)
                else:
                    x_t = x0
        out.append(np.clip((x_t.numpy() + 1) / 2, 0, 1))
    return np.concatenate(out)


_SCORER = []

In [ ]:
def evaluate(name, kind):
    d = run_dir(kind, name)
    res_path = os.path.join(d, "eval.json")
    if os.path.exists(res_path):
        print(open(res_path).read()); return
    net, fwd, opt, ema, ema_n, step, ckpt = make_state(name)
    ckpt.restore(tf.train.latest_checkpoint(os.path.join(d, "ckpt"))).expect_partial()
    for v, e in zip(net.trainable_variables, ema):
        v.assign(e)
    if not _SCORER:
        _SCORER.append(InceptionScorer())
    scorer = _SCORER[0]
    mu, sig, real_is = reference(scorer)
    results = {}
    for label, p, bc in predictors(name, net, fwd):
        t0 = time.time()
        imgs = generate(kind, p)
        samp = time.time() - t0
        f, pr = scorer(imgs)
        m, s = inception_score(pr)
        results[label] = dict(fid=fid_from_features(f, mu, sig), is_mean=m, is_std=s, params=int(net.count_params()),
                              block_compute=bc, sample_seconds=samp, steps=int(step.numpy()))
        np.save(os.path.join(d, f"samples_{label}.npy"), (imgs[:32] * 255).astype(np.uint8))
        print(f"[{kind}/{label}] FID {results[label]['fid']:.2f} IS {m:.2f}  sampling {samp:.0f}s", flush=True)
    json.dump(results, open(res_path, "w"), indent=2)


def report():
    rows = []
    for p in glob.glob(os.path.join(ROOT, "*", "*", "eval.json")):
        kind = p.split(os.sep)[-3]
        tr = json.load(open(os.path.join(os.path.dirname(p), "train_done.json")))
        loss = np.loadtxt(os.path.join(os.path.dirname(p), "loss.csv"), delimiter=",", ndmin=2)
        for label, r in json.load(open(p)).items():
            rows.append((kind, label, r, tr, loss[-1, 1]))
    for kind in ("gauss", "blur", "spec_eq", "spec_c2f"):
        print(f"\n=== {kind} ===")
        print(f"{'model':22s} {'params':>10s} {'ms/step':>8s} {'loss':>8s} {'FID':>8s} {'IS':>6s} {'blk%':>5s} {'samp s':>7s}")
        for k, label, r, tr, l in sorted([x for x in rows if x[0] == kind], key=lambda x: x[2]["fid"]):
            bc = "-" if r["block_compute"] is None else f"{r['block_compute'] * 100:.0f}"
            print(f"{label:22s} {r['params']:>10,} {tr['ms_per_step']:8.0f} {l:8.4f} {r['fid']:8.2f} {r['is_mean']:6.2f} {bc:>5s} {r['sample_seconds']:7.0f}")

## 10. New pipeline: spectral wavelet schedules
An orthonormal 2-level Haar transform splits the image into a coarse 8×8 band, a mid-detail band and a fine-detail band.
Each band gets the cosine log-SNR plus a shift computed from its power on real airplanes:
- `spec_eq`: every band crosses SNR = 1 at the same time
- `spec_c2f`: fine detail is destroyed even earlier, coarse layout survives longer

Haar is orthonormal, so the noise is still white in pixel space and the DDIM sampler works per coefficient.

In [ ]:
# ---------------------------------------------------------------- orthonormal 2-level Haar (Mallat layout)
def _haar1(x):
    a, b = x[:, 0::2, 0::2], x[:, 0::2, 1::2]
    c, d = x[:, 1::2, 0::2], x[:, 1::2, 1::2]
    ll, lh = (a + b + c + d) / 2, (a - b + c - d) / 2
    hl, hh = (a + b - c - d) / 2, (a - b - c + d) / 2
    return tf.concat([tf.concat([ll, lh], 2), tf.concat([hl, hh], 2)], 1)


def _ihaar1(w):
    h, ww = w.shape[1] // 2, w.shape[2] // 2
    ll, lh, hl, hh = w[:, :h, :ww], w[:, :h, ww:], w[:, h:, :ww], w[:, h:, ww:]
    a, b = (ll + lh + hl + hh) / 2, (ll - lh + hl - hh) / 2
    c, d = (ll + lh - hl - hh) / 2, (ll - lh - hl + hh) / 2
    top = tf.reshape(tf.stack([a, b], 3), (-1, h, 2 * ww, w.shape[3]))
    bot = tf.reshape(tf.stack([c, d], 3), (-1, h, 2 * ww, w.shape[3]))
    return tf.reshape(tf.stack([top, bot], 2), (-1, 2 * h, 2 * ww, w.shape[3]))


def haar2(x):
    w = _haar1(x)
    tl = _haar1(w[:, :16, :16])
    return tf.concat([tf.concat([tl, w[:, :16, 16:]], 2), w[:, 16:, :]], 1)


def ihaar2(w):
    tl = _ihaar1(w[:, :16, :16])
    return _ihaar1(tf.concat([tf.concat([tl, w[:, :16, 16:]], 2), w[:, 16:, :]], 1))


BAND = np.full((32, 32), 2, np.int32)     # 2 = level-1 detail (finest)
BAND[:16, :16] = 1                        # level-2 detail
BAND[:8, :8] = 0                          # coarse LL2


def band_shifts(kind):
    """log-SNR shift per band, from the data's per-band power."""
    x = tf.constant(load_data()[:2000])
    w = haar2(x).numpy()
    power = np.array([np.mean(w[:, BAND == b] ** 2) for b in range(3)])
    ref = np.mean(w ** 2)
    if kind == "spec_eq":
        return -np.log(power / ref)                 # every band crosses SNR=1 together
    return 0.5 * np.log(power / ref)                # spec_c2f: fine bands lose even earlier


def alpha_map(kind):
    """(T, 32, 32, 1) per-coefficient alpha_bar in wavelet layout."""
    ab_ = np.clip(ALPHA_BAR_NP.astype(np.float64), 1e-8, 1 - 1e-8)
    if kind == "gauss":
        return None
    logsnr = np.log(ab_ / (1 - ab_))[:, None]
    sh = band_shifts(kind)
    per_band = 1 / (1 + np.exp(-(logsnr + sh[None, :])))          # (T, 3)
    per_band[-1] = np.minimum(per_band[-1], 1e-4)                  # terminal state ~ pure noise
    m = per_band[:, BAND]                                          # (T, 32, 32)
    print(kind, "band shifts", np.round(sh, 2), "alpha_bar @t=500 per band", np.round(per_band[500], 3), flush=True)
    return tf.constant(m[..., None].astype(np.float32))


def noisy(kind, A, x0, t, noise):
    if A is None:
        a = tf.gather(ALPHA_BAR, t)[:, None, None, None]
        return tf.sqrt(a) * x0 + tf.sqrt(1 - a) * noise
    a = tf.gather(A, t)
    return ihaar2(tf.sqrt(a) * haar2(x0) + tf.sqrt(1 - a) * haar2(noise))

## 11. New architecture: SWL (Shared-Weight Wavelet Ladder)
- **Lossless Haar down/up** instead of strided convs
- **Two ladder passes** (depth 1, then depth 2) that **reuse the same per-level blocks**, plus a learned pass embedding
- **Noise-dependent gate per pass**, `g_p(t) = 1 + tanh(Dense(t))` (starts at 1)
- **adaLN-Zero** ResBlocks (scale/shift from the conditioning, zero-init output conv)
- attention at 8×8

Variants: `swl` (2.72M), `swl_wide` (80/160/160 ch, 3.74M), `swl_p3` (third pass), `swl_long` (same as `swl`, trained to 8k steps).

In [ ]:
# ---------------------------------------------------------------- SWL architecture
class AdaRes(layers.Layer):
    """ResBlock with adaLN-Zero conditioning (scale/shift from cond, zero-init output conv)."""
    def __init__(self, in_ch, ch):
        super().__init__()
        self.n1 = layers.GroupNormalization(8, center=False, scale=False)
        self.m1 = layers.Dense(2 * in_ch, kernel_initializer="zeros")
        self.c1 = layers.Conv2D(ch, 3, padding="same")
        self.n2 = layers.GroupNormalization(8, center=False, scale=False)
        self.m2 = layers.Dense(2 * ch, kernel_initializer="zeros")
        self.c2 = layers.Conv2D(ch, 3, padding="same", kernel_initializer="zeros")
        self.skip = layers.Conv2D(ch, 1) if in_ch != ch else None

    def call(self, x, c):
        s1, b1 = tf.split(self.m1(c)[:, None, None, :], 2, -1)
        h = self.c1(tf.nn.silu(self.n1(x) * (1 + s1) + b1))
        s2, b2 = tf.split(self.m2(c)[:, None, None, :], 2, -1)
        h = self.c2(tf.nn.silu(self.n2(h) * (1 + s2) + b2))
        return h + (self.skip(x) if self.skip is not None else x)


class HaarDown(layers.Layer):
    def __init__(self, ch):
        super().__init__(); self.proj = layers.Conv2D(ch, 1)
    def call(self, x):
        return self.proj(tf.nn.space_to_depth(x, 2) if False else _haar_s2d(x))


class HaarUp(layers.Layer):
    def __init__(self, ch):
        super().__init__(); self.ch = ch; self.proj = layers.Conv2D(4 * ch, 1)
    def call(self, x):
        return _haar_d2s(self.proj(x), self.ch)


def _haar_s2d(x):
    a, b = x[:, 0::2, 0::2], x[:, 0::2, 1::2]
    c, d = x[:, 1::2, 0::2], x[:, 1::2, 1::2]
    return tf.concat([(a + b + c + d) / 2, (a - b + c - d) / 2, (a + b - c - d) / 2, (a - b - c + d) / 2], -1)


def _haar_d2s(y, ch):
    ll, lh, hl, hh = tf.split(y, 4, -1)
    a, b = (ll + lh + hl + hh) / 2, (ll - lh + hl - hh) / 2
    c, d = (ll + lh - hl - hh) / 2, (ll - lh - hl + hh) / 2
    h, w = y.shape[1], y.shape[2]
    top = tf.reshape(tf.stack([a, b], 3), (-1, h, 2 * w, ch))
    bot = tf.reshape(tf.stack([c, d], 3), (-1, h, 2 * w, ch))
    return tf.reshape(tf.stack([top, bot], 2), (-1, 2 * h, 2 * w, ch))


class SWL(tf.keras.Model):
    """Shared-Weight Wavelet Ladder: passes of depth 1 then 2 reuse the same per-level blocks;
    each pass's update is scaled by a noise-dependent gate g_p(t) = 1 + tanh(.) (starts at 1)."""
    def __init__(self, ch=(64, 128, 128), time_dim=128, passes=(1, 2)):
        super().__init__()
        self.ch, self.passes, self.time_dim = ch, passes, time_dim
        cd = time_dim * 4
        self.time_mlp = tf.keras.Sequential([layers.Dense(cd, activation="swish"), layers.Dense(cd)])
        self.pass_emb = [self.add_weight(shape=(cd,), initializer="zeros", name=f"pass{p}") for p in range(len(passes))]
        self.gates = [layers.Dense(1, kernel_initializer="zeros") for _ in passes]
        self.stem = layers.Conv2D(ch[0], 3, padding="same")
        self.enc = [AdaRes(ch[l], ch[l]) for l in range(3)]
        self.dec = [AdaRes(ch[l], ch[l]) for l in range(2)]
        self.down = [HaarDown(ch[l + 1]) for l in range(2)]
        self.up = [HaarUp(ch[l]) for l in range(2)]
        self.merge = [layers.Conv2D(ch[l], 1) for l in range(2)]
        self.attn = PAttentionBlock(ch[2])
        self.out_norm = layers.GroupNormalization(8)
        self.out_conv = layers.Conv2D(3, 3, padding="same")

    def call(self, inputs, training=False):
        x, t = inputs
        te = self.time_mlp(timestep_embedding(t, self.time_dim))
        h0 = self.stem(x)
        for p, depth in enumerate(self.passes):
            c = te + self.pass_emb[p]
            h, skips = h0, []
            for l in range(depth):
                h = self.enc[l](h, c); skips.append(h); h = self.down[l](h)
            h = self.enc[depth](h, c)
            if depth == 2:
                h = self.attn(h)
            for l in reversed(range(depth)):
                h = self.up[l](h)
                h = self.merge[l](tf.concat([h, skips[l]], -1))
                h = self.dec[l](h, c)
            g = 1.0 + tf.tanh(self.gates[p](te))[:, :, None, None]
            h0 = h0 + g * (h - h0)
        return self.out_conv(tf.nn.silu(self.out_norm(h0)))


_ab_build = build


SWL_VARIANTS = {"swl": {}, "swl_long": {}, "swl_wide": dict(ch=(80, 160, 160)), "swl_p3": dict(passes=(1, 2, 2))}


def build(name):
    if name in SWL_VARIANTS:
        tf.keras.utils.set_random_seed(0)
        net = SWL(**SWL_VARIANTS[name])
        fwd = lambda x, t: net([x, t])
        fwd(tf.zeros((2, 32, 32, 3)), tf.constant([0, 999], tf.int32))
        return net, fwd
    return _ab_build(name)


# make_state / evaluate look up build() by name, so SWL variants are picked up

## 12. Training for the new pipeline (round-robin, identical stream)

In [ ]:
# ---------------------------------------------------------------- train (round-robin, identical stream)
def train_multi_novel(names, kind, steps, ckpt_every=500):
    A = alpha_map(kind)
    x_all = load_data()
    runs = []
    for name in names:
        d = run_dir(kind, name); os.makedirs(d, exist_ok=True)
        net, fwd, opt, ema, ema_n, step, ckpt = make_state(name)
        mgr = tf.train.CheckpointManager(ckpt, os.path.join(d, "ckpt"), max_to_keep=1)
        if mgr.latest_checkpoint:
            ckpt.restore(mgr.latest_checkpoint).expect_partial()

        def make(net=net, fwd=fwd, opt=opt, ema=ema, ema_n=ema_n):
            @tf.function(jit_compile=True)
            def f(x0, t, noise):
                xt = noisy(kind, A, x0, t, noise)
                with tf.GradientTape() as tape:
                    mse = tf.reduce_mean(tf.square(fwd(xt, t) - noise))
                g = tape.gradient(mse, net.trainable_variables)
                opt.apply_gradients(zip(g, net.trainable_variables))
                ema_n.assign_add(1.0)
                dd = tf.minimum(0.999, (1.0 + ema_n) / (10.0 + ema_n))
                for e, v in zip(ema, net.trainable_variables):
                    e.assign(dd * e + (1 - dd) * v)
                return mse
            return f
        runs.append(dict(name=name, d=d, net=net, step=step, mgr=mgr, fn=make(), losses=[], secs=0.0,
                         log=open(os.path.join(d, "loss.csv"), "a")))
        print(f"[{kind}/{name}] params {net.count_params():,}", flush=True)
    s = min(int(r["step"].numpy()) for r in runs)
    while s < steps:
        x, t, n = stream_batch(x_all, s)
        xt, tt, nt = tf.constant(x), tf.constant(t), tf.constant(n)
        for r in runs:
            if int(r["step"].numpy()) > s:
                continue
            t0 = time.time(); r["losses"].append(float(r["fn"](xt, tt, nt))); r["secs"] += time.time() - t0
            r["step"].assign(s + 1)
        s += 1
        if s % ckpt_every == 0 or s == steps:
            for r in runs:
                r["mgr"].save(); m = float(np.mean(r["losses"][-ckpt_every:]))
                r["log"].write(f"{s},{m:.6f}\n"); r["log"].flush()
                print(f"[{kind}/{r['name']}] step {s} loss {m:.4f} {r['secs'] / max(1, len(r['losses'])) * 1000:.0f} ms/step", flush=True)
    for r in runs:
        json.dump(dict(params=int(r["net"].count_params()), steps=int(r["step"].numpy()),
                       ms_per_step=r["secs"] / max(1, len(r["losses"])) * 1000), open(os.path.join(r["d"], "train_done.json"), "w"))

## 13. Sampling with per-coefficient alphas

In [ ]:
# ---------------------------------------------------------------- eval (DDIM generalised to per-coefficient alpha)
def generate_spectral(kind, predict):
    A = alpha_map(kind)
    rng = np.random.default_rng(EVAL_SEED)
    idx = SAMPLE_TS[::-1]
    out = []
    for start in range(0, N_EVAL, EVAL_BATCH):
        n = min(EVAL_BATCH, N_EVAL - start)
        x_t = tf.constant(rng.standard_normal((n, 32, 32, 3)), tf.float32)
        for i, t in enumerate(idx):
            tp = idx[i + 1] if i + 1 < len(idx) else -1
            a_t = A[int(t)][None]; a_p = A[int(tp)][None] if tp >= 0 else tf.ones_like(a_t)
            eps_w = haar2(predict(x_t, int(t)))
            xw = haar2(x_t)
            x0 = tf.clip_by_value(ihaar2((xw - tf.sqrt(1 - a_t) * eps_w) / tf.sqrt(a_t)), -1, 1)
            x_t = ihaar2(tf.sqrt(a_p) * haar2(x0) + tf.sqrt(1 - a_p) * eps_w)
        out.append(np.clip((x_t.numpy() + 1) / 2, 0, 1))
    return np.concatenate(out)


_orig_generate = generate
generate = lambda kind, predict: _orig_generate(kind, predict) if kind in ("gauss", "blur") else generate_spectral(kind, predict)



## 14. How to run
On the DGX these ran as SLURM jobs (see `run_all.sbatch`, `novel.sbatch`). From this notebook, inside a GPU session:

In [ ]:
RUN = False      # set True to actually train / evaluate

if RUN:
    # main comparison, Gaussian and blur (4,000 steps each)
    MAIN = ["original_unet", "unet_v1", "ladder_v1", "unet_v2", "ladder_v2", "ladder_rev_v2",
            "routed_dense", "routed_fixed", "routed_learned"]
    for kind in ("gauss", "blur"):
        train_multi(MAIN, kind, 4000)
        for m in MAIN:
            evaluate(m, kind)

    # new pipeline: SWL + spectral schedules
    train_multi_novel(["swl"], "gauss", 4000)
    for kind in ("spec_eq", "spec_c2f"):
        train_multi_novel(["original_unet", "ladder_v2", "swl"], kind, 4000)

    # improvement round
    train_multi_novel(["swl_wide", "swl_p3"], "gauss", 4000)
    train_multi_novel(["swl_long"], "gauss", 8000)   # the DGX run continued swl's 4k checkpoint
    for m in ["swl", "swl_wide", "swl_p3", "swl_long"]:
        evaluate(m, "gauss")

## 15. Results

In [ ]:
RESULTS = os.path.abspath("../results")  # BTP-Diffusion-dgx/results/
ROOT = RESULTS
report()

### Summary (FID on 5,000 samples, lower is better; real airplanes IS 4.91)

| Gaussian | params | FID | IS |
|---|---|---|---|
| **swl_long** (8k steps) | 2.72M | **72.0** | **4.42** |
| swl_wide | 3.74M | 87.4 | 3.90 |
| **swl** | 2.72M | **92.8** | 3.74 |
| swl_p3 | 2.72M | 100.5 | 3.47 |
| original_unet | 1.66M | 128.5 | 3.17 |
| ladder_v2 | 3.21M | 128.9 | 2.93 |
| ladder_rev_v2 | 3.21M | 146.5 | 2.80 |
| unet_v2 | 2.37M | 149.9 | 2.78 |
| ladder_v1 | 2.35M | 176.3 | 2.50 |
| routed_dense | 3.94M | 192.8 | 2.38 |
| unet_v1 | 1.54M | 209.4 | 2.25 |
| routed_fixed | 3.94M | 228.3 | 2.11 |

**Blur + noise:** ladder_rev_v2 **136.6** · ladder_v2 141.3 · unet_v2 151.8 · ladder_v1 154.7 · original_unet 160.4 · unet_v1 162.6 · routed_dense 165.2 · routed_fixed 224.1

**Spectral schedules** (FID): swl 92.8 → 111.7 (eq) / 113.2 (c2f); original_unet 128.5 → 149.9 / 182.0; ladder_v2 128.9 → 142.4 / 196.3. They hurt every model; SWL is the most robust.

**Takeaways:** SWL beats the original UNet by about 28% at equal training steps (one seed; ~2–3× slower sampling); the fixed bricks confirm the downsampling hypothesis; the ladder helps on good bricks; routing by noise level and the spectral schedules didn't help.

In [ ]:
from IPython.display import Image
Image(os.path.join(RESULTS, "samples_top.png"))